In [1]:

import zipfile, io, urllib.request
import pandas as pd

In [2]:
url = "https://archive.ics.uci.edu/static/public/317/grammatical+facial+expressions.zip"
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
nomes = sorted(n for n in z.namelist() if n.endswith("_datapoints.txt") and "__MACOSX" not in n)
todos = set(z.namelist())

partes = []
for n in nomes:
    video = n.split("/")[-1].replace("_datapoints.txt", "")
    usuario, expressao = video.split("_", 1)
    alvo_nome = next(c for c in (n.replace("_datapoints.txt", "_targets.txt"),
                                 n.replace("doubt_question_datapoints.txt", "doubts_question_targets.txt")) if c in todos)
    dados = pd.read_csv(z.open(n), sep=r"\s+")
    dados = dados.rename(columns={dados.columns[0]: "timestamp"})
    alvo = pd.read_csv(z.open(alvo_nome), header=None)[0].rename("target")
    extra = pd.DataFrame({"usuario": usuario, "expressao": expressao}, index=dados.index)
    partes.append(pd.concat([dados, extra, alvo], axis=1))

df = pd.concat(partes, ignore_index=True)
df

,timestamp,0x,0y,0z,1x,1y,1z,2x,2y,2z,...,97z,98x,98y,98z,99x,99y,99z,usuario,expressao,target
0,1.390385e+09,244.672,208.182,895,240.949,205.708,890,235.994,205.018,883,...,944,291.722,189.204,0,296.547,192.722,1019,a,affirmative,0
1,1.390385e+09,265.953,195.731,1016,262.556,193.301,1007,258.243,192.547,1001,...,1069,308.776,180.199,0,312.501,184.498,0,a,affirmative,0
2,1.390385e+09,272.755,192.786,1060,269.481,190.460,1056,265.371,189.756,1053,...,1114,314.125,177.992,1140,317.529,182.324,1179,a,affirmative,0
3,1.390385e+09,281.890,190.533,1129,278.861,188.023,1129,274.899,187.213,1136,...,1187,322.016,180.029,0,325.041,184.442,1252,a,affirmative,0
4,1.390385e+09,283.933,193.201,1159,280.955,190.877,1159,277.180,190.196,1167,...,1216,321.384,181.259,1225,324.488,185.400,1284,a,affirmative,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27931,1.391186e+09,327.201,227.957,1265,324.667,225.769,1270,321.280,224.810,1270,...,1289,357.639,217.601,1299,361.807,220.073,1304,b,yn_question,0
27932,1.391186e+09,327.201,227.957,1265,324.667,225.769,1275,321.280,224.810,1275,...,1294,357.639,217.601,1299,361.807,220.073,1309,b,yn_question,0
27933,1.391186e+09,327.198,227.959,1265,324.664,225.769,1275,321.277,224.807,1275,...,1294,357.648,217.620,1299,361.816,220.095,1304,b,yn_question,0
27934,1.391186e+09,327.198,227.959,1265,324.664,225.769,1275,321.277,224.807,1270,...,1294,357.648,217.620,1299,361.816,220.095,1304,b,yn_question,0


In [3]:
# expressao yn_question, sobrancelhas (x e y) e iris
yn = df[df["expressao"] == "yn_question"]

colunas_sobrancelha = [f"{i}{eixo}" for i in range(16, 36) for eixo in "xy"]  #pontos 16 a 35, so x e y
colunas_iris = ["87x", "87y", "88x", "88y"]

dados_uso = yn[["usuario"] + colunas_sobrancelha + colunas_iris + ["target"]]
print(dados_uso.shape)
print(dados_uso.groupby("usuario")["target"].agg(quadros="size", positivos="sum", proporcao="mean").round(3))
dados_uso.head()

(3128, 46)
         quadros  positivos  proporcao
usuario                               
a           1390        532      0.383
b           1738        715      0.411


,usuario,16x,16y,17x,17y,18x,18y,19x,19y,20x,...,33y,34x,34y,35x,35y,87x,87y,88x,88y,target
12220,a,301.241,152.241,291.578,150.416,280.534,150.613,274.254,155.104,270.927,...,154.452,334.641,154.608,324.912,156.006,291.034,166.336,335.795,165.567,0
12221,a,301.172,177.569,293.055,176.296,283.928,177.102,278.560,181.142,275.692,...,179.743,329.387,179.697,321.190,180.862,292.184,191.468,331.338,190.242,0
12222,a,300.281,176.207,292.608,175.017,284.064,175.804,279.002,179.637,276.228,...,178.489,327.046,178.413,319.487,179.463,291.760,190.227,329.110,189.185,0
12223,a,300.337,175.876,292.869,174.784,284.568,175.624,279.668,179.389,277.013,...,178.151,325.921,178.030,318.618,178.996,292.073,189.025,327.836,188.016,0
12224,a,300.175,175.812,292.760,174.733,284.526,175.538,279.736,179.314,277.165,...,178.192,325.481,178.039,318.285,178.936,292.115,188.873,327.310,188.033,0


In [4]:
# ausentes, duplicados e atributos constantes

coord = [c for c in df.columns if c[:-1].isdigit() and c[-1] in "xyz"]
print("atributos de coordenadas:", len(coord))
print("ausentes:", int(df[coord + ["timestamp", "target"]].isna().sum().sum()))
print("linhaas duplicadas:", int(df.duplicated(subset=coord).sum()))
print("atributos constantes:", [c for c in coord if df[c].nunique() == 1])

atributos de coordenadas: 300
ausentes: 0
linhaas duplicadas: 0
atributos constantes: []


In [5]:
# zeros nas coordenadas

zcols = [c for c in coord if c.endswith("z")]
xycols = [c for c in coord if not c.endswith("z")]
print("zeros em z:", int((df[zcols] == 0).sum().sum()), "de", df[zcols].size)
print("zeros em x ou y:", int((df[xycols] == 0).sum().sum()))

zeros em z: 117304 de 2793600
zeros em x ou y: 0


In [6]:
# correlacao entre os atributos

import numpy as np
corr = df[coord].corr().abs()
r = corr.values[np.triu_indices_from(corr, k=1)]
print("pares com |r| > 0.9:", int((r > 0.9).sum()), "de", len(r), f"({(r > 0.9).mean():.1%})")


pares com |r| > 0.9: 9892 de 44850 (22.1%)


In [7]:
# balanceamento por usuário e expressao

df.groupby(["usuario", "expressao"])["target"].agg(quadros="size", positivos="sum", proporcao="mean").round(3)

quadros  positivos  proporcao
usuario expressao                                    
a       affirmative        1062        414      0.390
        conditional        1907        548      0.287
        doubt_question     1312        491      0.374
        emphasis           1403        330      0.235
        negative           1124        528      0.470
        relative           2330        644      0.276
        topics             1796        360      0.200
        wh_question        1286        609      0.474
        yn_question        1390        532      0.383
b       affirmative        1074        528      0.492
        conditional        2034        589      0.290
        doubt_question     1497        780      0.521
        emphasis           1344        531      0.395
        negative           1582        712      0.450
        relative           1904        550      0.289
        topics             1825        467      0.256
        wh_question        1328        549      0.413
        yn_question        1738        715      0.411

In [8]:
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
BLOCK = 30
N_BOOT = 1000
SOBRANCELHAS = list(range(16, 36))
IRIS_E, IRIS_D = 87, 88

In [9]:
# funcao que monta os atributos
# as 3 versões dos dados: bruto, normalizado e altura_media

def atributos(d, tipo):
  # monta a matriz de atributos
  # no eixo y da imagem, valores maiores estão mais abaixo
  ex = d[[f"{i}x" for i in SOBRANCELHAS]].values
  ey = d[[f"{i}y" for i in SOBRANCELHAS]].values
  if tipo == "bruto":
    return np.hstack([ex, ey])
  ox = (d[f"{IRIS_E}x"].values + d[f"{IRIS_D}x"].values) / 2
  oy = (d[f"{IRIS_E}y"].values + d[f"{IRIS_D}y"].values) / 2
  escala = np.hypot(d[f"{IRIS_E}x"].values - d[f"{IRIS_D}x"].values,
                    d[f"{IRIS_E}y"].values - d[f"{IRIS_D}y"].values)
  nx = (ex - ox[:, None]) / escala[:, None]
  ny = (oy[:, None] - ey) / escala[:, None]
  if tipo == "normalizado":
    return np.hstack([nx, ny])
  if tipo == "altura_media":
    return ny.mean(axis=1, keepdims=True)
  raise ValueError(tipo)

In [10]:
MODELOS = {
    "arvore_prof3": lambda: DecisionTreeClassifier(max_depth=3, random_state=SEED),
    "reg_logistica": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED))
}

def ic_bootstrap(y, p, metrica, rng):
  # IC de 95% por bootstrap em blocos de quadros consecutivos
  n = len(y)
  blocos = [np.arange(s, min(s + BLOCK, n)) for s in range(0, n, BLOCK)]
  valores = []
  for _ in range(N_BOOT):
    sorteio = rng.integers(0, len(blocos), len(blocos))
    idx = np.concatenate([blocos[k] for k in sorteio])
    valores.append(metrica(y[idx], p[idx]))
  return np.percentile(valores, [2.5, 97.5])

def avaliar(y, p, rng):
  f1_inf, f1_sup = ic_bootstrap(y, p, lambda a, b: f1_score(a, b, zero_division=0), rng)
  mcc_inf, mcc_sup = ic_bootstrap(y, p, matthews_corrcoef, rng)
  return {"acuracia": accuracy_score(y, p),
          "precisao": precision_score(y, p, zero_division=0),
          "revocacao": recall_score(y, p, zero_division=0),
          "F1": f1_score(y, p, zero_division=0), "F1_ic_inf": f1_inf, "F1_ic_sup": f1_sup,
          "MCC": matthews_corrcoef(y, p), "MCC_ic_inf": mcc_inf, "MCC_ic_sup": mcc_sup}


In [11]:
rng = np.random.default_rng(SEED)
linhas = []
for treino_u, teste_u in [("a", "b"), ("b", "a")]:
  tr = dados_uso[dados_uso["usuario"] == treino_u]
  te = dados_uso[dados_uso["usuario"] == teste_u]
  ytr, yte = tr["target"].values, te["target"].values
  base = {"treino": treino_u.upper(), "teste": teste_u.upper()}
  zero = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(ytr), 1)), ytr)
  linhas.append({**base, "atributos": "-", "modelo": "baseline_majoritaria",
                 **avaliar(yte, zero.predict(np.zeros((len(yte), 1))), rng)})
  um = DummyClassifier(strategy="constant", constant=1).fit(np.zeros((len(ytr), 1)), ytr)
  linhas.append({**base, "atributos": "-", "modelo": "baseline_sempre_positivo",
                 **avaliar(yte, um.predict(np.zeros((len(yte), 1))), rng)})
  for tipo in ["bruto", "normalizado", "altura_media"]:
    Xtr, Xte = atributos(tr, tipo), atributos(te, tipo)
    for nome, criar in MODELOS.items():
      modelo = criar().fit(Xtr, ytr)
      linhas.append({**base, "atributos": tipo, "modelo": nome, **avaliar(yte, modelo.predict(Xte), rng)})

res = pd.DataFrame(linhas)
res.to_csv("resultados_yn.csv", index=False)
print("resultados (teste em um usuário que não foi visto no treino):")
display(res.round(3))

resultados (teste em um usuário que não foi visto no treino):


,treino,teste,atributos,modelo,acuracia,precisao,revocacao,F1,F1_ic_inf,F1_ic_sup,MCC,MCC_ic_inf,MCC_ic_sup
0,A,B,-,baseline_majoritaria,0.589,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,A,B,-,baseline_sempre_positivo,0.411,0.411,1.000,0.583,0.503,0.652,0.000,0.000,0.000
2,A,B,bruto,arvore_prof3,0.411,0.411,1.000,0.583,0.503,0.649,0.000,0.000,0.000
3,A,B,bruto,reg_logistica,0.575,0.487,0.621,0.546,0.462,0.628,0.161,0.026,0.293
4,A,B,normalizado,arvore_prof3,0.597,0.527,0.206,0.296,0.141,0.441,0.103,-0.077,0.254
5,A,B,normalizado,reg_logistica,0.509,0.455,0.985,0.623,0.552,0.681,0.254,0.122,0.361
6,A,B,altura_media,arvore_prof3,0.543,0.468,0.818,0.596,0.510,0.670,0.185,0.031,0.325
7,A,B,altura_media,reg_logistica,0.700,0.914,0.298,0.449,0.317,0.566,0.402,0.307,0.502
8,B,A,-,baseline_majoritaria,0.617,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
9,B,A,-,baseline_sempre_positivo,0.383,0.383,1.000,0.554,0.471,0.630,0.000,0.000,0.000


In [12]:
# media das duas direcoes

resumo = res.groupby(["atributos", "modelo"], sort=False)[["acuracia", "F1", "MCC"]].mean().round(3)
print("média das duas direções (A->B e B->A):")
display(resumo)

média das duas direções (A->B e B->A):


acuracia     F1    MCC
atributos    modelo                                          
-            baseline_majoritaria         0.603  0.000  0.000
             baseline_sempre_positivo     0.397  0.568  0.000
bruto        arvore_prof3                 0.514  0.291  0.000
             reg_logistica                0.596  0.273  0.081
normalizado  arvore_prof3                 0.607  0.148  0.051
             reg_logistica                0.563  0.311  0.127
altura_media arvore_prof3                 0.578  0.345  0.109
             reg_logistica                0.660  0.230  0.231

In [13]:
from sklearn.model_selection import train_test_split

linhas_ctrl = []
for tipo in ["bruto", "normalizado", "altura_media"]:
  X, y = atributos(dados_uso, tipo), dados_uso["target"].values
  Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
  for nome, criar in MODELOS.items():
    p = criar().fit(Xtr, ytr).predict(Xte)
    mcc_entre = res[(res["atributos"] == tipo) & (res["modelo"] == nome)].set_index("treino")["MCC"]
    linhas_ctrl.append({"atributos": tipo, "modelo": nome,
                        "MCC A->B": mcc_entre["A"], "MCC B->A": mcc_entre["B"],
                        "MCC controle 70/30": matthews_corrcoef(yte, p),
                        "acuracia controle": accuracy_score(yte, p)})
print("controle: divisão aleatória 70/30 X avaliação entre participantes, na MESMA representação")
display(pd.DataFrame(linhas_ctrl).round(3))

controle: divisão aleatória 70/30 X avaliação entre participantes, na MESMA representação


,atributos,modelo,MCC A->B,MCC B->A,MCC controle 70/30,acuracia controle
0,bruto,arvore_prof3,0.000,0.000,0.589,0.802
1,bruto,reg_logistica,0.161,0.000,0.575,0.798
2,normalizado,arvore_prof3,0.103,0.000,0.738,0.875
3,normalizado,reg_logistica,0.254,0.000,0.764,0.887
4,altura_media,arvore_prof3,0.185,0.034,0.377,0.705
5,altura_media,reg_logistica,0.402,0.059,0.286,0.673


In [14]:
# altura media das sobrancelhas por usuario e por classe
from sklearn.metrics import roc_auc_score

h = pd.Series(atributos(dados_uso, "altura_media")[:, 0], index=dados_uso.index)
display(h.groupby([dados_uso["usuario"], dados_uso["target"]]).agg(["mean", "std", "count"]).round(3))

for u in ["a", "b"]:
  g = dados_uso["usuario"] == u
  x1, x0 = h[g & (dados_uso["target"] == 1)], h[g & (dados_uso["target"] == 0)]
  sp = np.sqrt(((len(x1) - 1) * x1.var() + (len(x0) - 1) * x0.var()) / (len(x1) + len(x0) - 2))
  print(f"usuario {u.upper()}: diferença de médias {x1.mean() - x0.mean():+.3f} | d de Cohen {(x1.mean() - x0.mean()) / sp:+.2f} | AUC {roc_auc_score(dados_uso.loc[g, 'target'], h[g]):.3f}")

mean    std  count
usuario target                     
a       0       0.250  0.013    858
        1       0.251  0.017    532
b       0       0.284  0.028   1023
        1       0.314  0.045    715

usuario A: diferença de médias +0.001 | d de Cohen +0.08 | AUC 0.492
usuario B: diferença de médias +0.030 | d de Cohen +0.83 | AUC 0.728


In [15]:
import sys, sklearn
print("Python", sys.version.split()[0])
print("pandas", pd.__version__,"| numpy", np.__version__,"| scikit-learn", sklearn.__version__)

Python 3.13.18
pandas 2.2.3 | numpy 2.4.6 | scikit-learn 1.6.1
